# Resume saved NimbusPay evaluation

Run only when **free Colab T4** access is available. This notebook loads completed Drive artifacts, finishes the baseline and exports test predictions. It never calls training. Completed predictions are preserved after checking IDs; incomplete writes are rejected for inspection. The separate notebook.ipynb remains the fresh-training reproduction path.

In [ ]:
import os
os.environ['USE_TF']='0'
os.environ['USE_FLAX']='0'
os.environ['TOKENIZERS_PARALLELISM']='false'
%pip install -q torch==2.11.0 transformers==4.57.6 peft==0.18.1 accelerate==1.12.0 bitsandbytes==0.49.1 datasets==4.4.2 huggingface-hub==0.36.0 tokenizers==0.22.2 safetensors==0.7.0


In [ ]:
from pathlib import Path
import torch, sys, zipfile, shutil
from google.colab import drive, files
assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(0), 'Free T4 required; do not use the CPU recovery runtime for inference'
ROOT=Path('/content/nimbuspay_resume')
ROOT.mkdir(exist_ok=True)
os.chdir(ROOT);sys.path.insert(0,str(ROOT))
drive.mount('/content/drive')
BACKUP=Path('/content/drive/MyDrive/nimbuspay-ticket-triage-private')
OUT=BACKUP/'artifacts'
assert OUT.exists()
uploaded=files.upload()
archive=next(name for name in uploaded if name.endswith('.zip'))
with zipfile.ZipFile(archive) as z:
    for name in ['candidate_pack/SCHEMA.md','candidate_pack/data/dev.jsonl','candidate_pack/data/test_inputs.jsonl']:
        path=ROOT/'private'/name
        path.parent.mkdir(parents=True,exist_ok=True)
        path.write_bytes(z.read(name))
del uploaded
DATA=ROOT/'private/candidate_pack/data'
SUBMISSION=BACKUP/'submission'


In [ ]:
%%writefile audit.py
"""Training-only, evidence-based label repair. Never imported by inference."""
import argparse
import copy
import csv
import hashlib
import json
import re
from collections import Counter
from datetime import date, timedelta
from decimal import Decimal
from pathlib import Path

KEYS = ('category', 'priority', 'amount_paise', 'txn_date', 'txn_id', 'channel', 'language', 'needs_human')
CATEGORIES = {'payment_failed', 'refund', 'fraud', 'account_access', 'kyc', 'offers', 'other'}
NO_TRANSACTION = {'account_access', 'kyc', 'other'}
LEGAL = re.compile(r'\b(RBI|ombudsman|consumer court|lawyer|legal notice|legal action)\b', re.I)
REPEAT = re.compile(r'third time|already written \d+ times|contacted support about this before|again and again|teesri baar|pehle bhi complaint|baar baar complaint', re.I)
ID_PATTERN = re.compile(r'\bnp[\s\-]*\d(?:[\s\-]*\d){9}(?!\d)', re.I)
NUM = r'\d[\d,]*(?:\.\d+)?'
MONEY = re.compile(rf'(?:₹|\bINR\b|\bRs\.?)\s*(?P<a>{NUM})\s*(?P<as>lakhs?|k\b)?|(?<![\w.])(?P<b>{NUM})\s*(?P<bs>lakhs?\b|k\b|rupees\b|rupaye\b)', re.I)
DISTRACTOR = re.compile(r'account balance|daily limit|balance is|account mein abhi|fee(?:s)?\b|limit is', re.I)
MONTHS = {m: i for i, m in enumerate(('jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec'), 1)}
MONTH = r'(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)'


def load(path):
    return [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]


def dump_jsonl(path, rows):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in rows))


def body(text):
    """Evidence ignores Received/Source metadata and earlier quoted emails."""
    return '\n'.join(line for line in text.splitlines()[2:] if not line.lstrip().startswith('>'))


def amount_evidence(text):
    candidates = []
    for match in MONEY.finditer(text):
        prefix = text[max(0, match.start()-70):match.start()].rsplit('. ', 1)[-1].rsplit('\n', 1)[-1]
        if DISTRACTOR.search(prefix):
            continue
        value = Decimal((match['a'] or match['b']).replace(',', ''))
        suffix = (match['as'] or match['bs'] or '').lower()
        multiplier = 100000 if suffix.startswith('lakh') else 1000 if suffix == 'k' else 1
        paise = value * multiplier * 100
        if paise != paise.to_integral_value():
            raise ValueError('amount_has_fractional_paise')
        candidates.append((int(paise), match.group(), match.span()))
    values = {v for v, _, _ in candidates}
    if len(values) > 1:
        raise ValueError('ambiguous_disputed_amount')
    return (next(iter(values)) if values else None), candidates


def resolved_date(text, received):
    text = re.sub(r'\b(\d{1,2})(?:st|nd|rd|th)\b', r'\1', text, flags=re.I)
    found = []
    for m in re.finditer(r'(?<![\d/\-])(\d{1,2})[/-](\d{1,2})[/-](\d{4}|\d{2})(?!\d)', text):
        day, month, year = map(int, m.groups())
        found.append(date(year if year >= 100 else 2000+year, month, day))
    for pattern, order in [(rf'\b(\d{{1,2}})\s+({MONTH})(?:\s+(\d{{4}}))?\b', 'dm'), (rf'\b({MONTH})\s+(\d{{1,2}})(?:,?\s+(\d{{4}}))?\b', 'md')]:
        for m in re.finditer(pattern, text, re.I):
            day = int(m[1] if order == 'dm' else m[2])
            month = MONTHS[(m[2] if order == 'dm' else m[1])[:3].lower()]
            year = int(m[3]) if m[3] else received.year
            parsed = date(year, month, day)
            if not m[3] and parsed > received:
                parsed = date(year-1, month, day)
            found.append(parsed)
    # Longer relative phrases take precedence over substrings such as yesterday.
    relative = text.lower()
    if re.search(r'day before yesterday|\bparso\b', relative):
        found.append(received-timedelta(days=2))
        relative = re.sub(r'day before yesterday|\bparso\b', '', relative)
    for m in re.finditer(r'\b(\d+)\s+(?:days? ago|din pehle)\b', relative):
        found.append(received-timedelta(days=int(m[1])))
    if re.search(r'\byesterday\b|\bkal\b', relative):
        found.append(received-timedelta(days=1))
    if re.search(r'\btoday\b|\baaj\b', relative):
        found.append(received)
    values = {d.isoformat() for d in found}
    if len(values) > 1:
        raise ValueError('ambiguous_transaction_date')
    return next(iter(values)) if values else None


def priority(category, amount, text):
    if category == 'fraud' or (amount is not None and amount >= 5000000) or LEGAL.search(text):
        return 'P1'
    if category == 'account_access' or (category in {'payment_failed', 'refund'} and amount is not None and amount >= 500000):
        return 'P2'
    return 'P3'


def validate(label):
    assert set(label) == set(KEYS)
    assert label['category'] in CATEGORIES
    assert label['priority'] in {'P1','P2','P3'}
    assert label['language'] in {'en','hi-en'}
    assert type(label['needs_human']) is bool
    assert label['amount_paise'] is None or type(label['amount_paise']) is int
    assert label['channel'] in {None,'upi','card','netbanking','wallet'}
    assert label['txn_id'] is None or re.fullmatch(r'NP\d{10}', label['txn_id'])
    if label['txn_date'] is not None:
        assert date.fromisoformat(label['txn_date']).isoformat() == label['txn_date']
    if label['category'] in NO_TRANSACTION:
        assert all(label[k] is None for k in ('amount_paise','txn_date','txn_id','channel'))
    assert label['priority'] != 'P1' or label['needs_human'] is True


def repair(row):
    original = json.loads(row['messages'][-1]['content'])
    label = copy.deepcopy(original)
    text = body(row['messages'][1]['content'])
    if re.fullmatch(r'\s*(?:test(?: test)?(?: 123)?|asdf asdf|hi|hello|hey|[.!?]*)\s*', text, re.I):
        raise ValueError('empty_or_test_ticket_unrelated_label')
    reasons = []
    if 'prio' in label:
        label['priority'] = {'low':'P3','medium':'P2','high':'P1'}[label.pop('prio')]
        reasons.append('legacy_schema_prio_to_priority')
    if label['category'] not in CATEGORIES:
        raise ValueError('unknown_category')
    if label['category'] in NO_TRANSACTION:
        for k in ('amount_paise','txn_date','txn_id','channel'):
            if label[k] is not None:
                reasons.append('non_transaction_category_' + k)
            label[k] = None
    else:
        received = date.fromisoformat(row['messages'][1]['content'].splitlines()[0].split()[1])
        amount, _ = amount_evidence(text)
        parsed_date = resolved_date(text, received)
        ids = {re.sub(r'[^A-Z0-9]', '', m.group().upper()) for m in ID_PATTERN.finditer(text)}
        if len(ids) > 1:
            raise ValueError('ambiguous_nimbuspay_transaction_id')
        mode_matches = []
        mode_text = re.sub(r'\b(?:scratch|srcatch)\s+card\b', 'reward', text, flags=re.I)
        for mode, pattern in [('netbanking',r'\bnetbanking\b|\binternet banking\b'),('upi',r'\bupi\b'),('wallet',r'\bwallet\b'),('card',r'\bcard\b|\bvisa\b|\bmastercard\b')]:
            if re.search(pattern, mode_text, re.I):
                mode_matches.append(mode)
        # Multiple modes require semantic review rather than guessing.
        if len(mode_matches) > 1:
            raise ValueError('ambiguous_transaction_channel')
        values = {'amount_paise':amount,'txn_date':parsed_date,'txn_id':next(iter(ids)) if ids else None,'channel':mode_matches[0] if mode_matches else None}
        for key, value in values.items():
            if type(label[key]) is not type(value) or label[key] != value:
                reasons.append('ticket_evidence_' + key)
                label[key] = value
    expected = priority(label['category'], label['amount_paise'], text)
    if label['priority'] != expected:
        reasons.append('schema_priority_order_and_thresholds')
        label['priority'] = expected
    human = expected == 'P1' or bool(REPEAT.search(text))
    if label['needs_human'] is not human:
        reasons.append('schema_needs_human_repeat_or_p1')
        label['needs_human'] = human
    validate(label)
    return {k:label[k] for k in KEYS}, reasons


def clean(rows, out):
    assert all(row['id'].startswith('tr-') for row in rows), 'Only original training rows are accepted'
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)
    groups = {}
    retained, log, evidence = [], [], []
    counters = Counter()
    for row in sorted(rows, key=lambda r:r['id']):
        user = row['messages'][1]['content']
        normalized = re.sub(r'\s+', ' ', user).strip()
        try:
            label, reasons = repair(row)
            if normalized in groups:
                prior_id, prior_label = groups[normalized]
                if label != prior_label:
                    raise ValueError('conflicting_normalized_duplicate')
                reason = 'duplicate_ticket_keep_' + prior_id
                log.append({'id':row['id'],'action':'drop','reason':reason})
                counters['duplicate'] += 1
                evidence.append({'id':row['id'],'drop_reason':reason,'kept_id':prior_id,'ticket':user,'before':row['messages'][-1]['content'],'corrected_label_agrees':True})
                continue
            groups[normalized] = (row['id'], label)
            cleaned = copy.deepcopy(row)
            cleaned['messages'][-1]['content'] = json.dumps(label, ensure_ascii=False)
            retained.append(cleaned)
            if reasons:
                log.append({'id':row['id'],'action':'fix','reason':';'.join(reasons)})
                counters.update(reasons)
                evidence.append({'id':row['id'],'before':row['messages'][-1]['content'],'after':label,'ticket':user,'rules':reasons})
        except (json.JSONDecodeError, ValueError, AssertionError, KeyError) as exc:
            reason = 'truncated_or_invalid_json' if isinstance(exc,json.JSONDecodeError) else str(exc) or 'schema_validation_failure'
            log.append({'id':row['id'],'action':'drop','reason':reason})
            counters[reason] += 1
            evidence.append({'id':row['id'],'drop_reason':reason,'ticket':user,'before':row['messages'][-1]['content']})
    dump_jsonl(out/'cleaned_train.jsonl',retained)
    dump_jsonl(out/'raw_training_evidence.jsonl',evidence)
    with (out/'cleaning_log.csv').open('w',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=['id','action','reason']);writer.writeheader();writer.writerows(log)
    summary={'input_rows':len(rows),'retained_rows':len(retained),'actions':dict(Counter(r['action'] for r in log)),'findings':dict(counters),'long_ticket_count':sum(len(r['messages'][1]['content'])>1500 for r in rows),'retained_long_ticket_count':sum(len(r['messages'][1]['content'])>1500 for r in retained),'training_sha256':hashlib.sha256(json.dumps(rows,ensure_ascii=False,sort_keys=True).encode()).hexdigest()}
    (out/'audit_summary.json').write_text(json.dumps(summary,indent=2))
    return retained,summary


def augment(rows, seed=42):
    """Equivalent money/id notation observed in train; no new labels."""
    import random
    rng = random.Random(seed)
    result = copy.deepcopy(rows)
    eligible = [i for i,r in enumerate(rows) if any(json.loads(r['messages'][-1]['content'])[key] is not None for key in ('txn_id','amount_paise'))]
    rng.shuffle(eligible)
    selected = eligible[:round(.2*len(eligible))]
    for i in selected:
        row = result[i]
        label = json.loads(row['messages'][-1]['content'])
        text = row['messages'][1]['content']
        if label['txn_id'] is not None:
            digits = label['txn_id'][2:]
            variants = ['np'+digits, 'NP-'+digits, 'NP '+digits[:5]+' '+digits[5:]]
            match = next(m for m in ID_PATTERN.finditer(text) if re.sub(r'[^A-Z0-9]','',m.group().upper()) == label['txn_id'])
            replacement = next(v for v in variants if v != match.group())
            row['messages'][1]['content'] = text[:match.start()] + replacement + text[match.end():]
        else:
            _, candidates = amount_evidence(body(text))
            original = candidates[0][1].rstrip()
            value = Decimal(label['amount_paise'])/100
            variants = [f'INR {value:.2f}', f'₹{value:.2f}', f'Rs {value:.2f}']
            replacement = next(v for v in variants if v != original)
            row['messages'][1]['content'] = text.replace(original,replacement,1)
        repaired,_ = repair(row)
        assert repaired == label, 'Augmentation must preserve every label field'
    assert len(result) == len(rows)
    return result, len(selected)


if __name__ == '__main__':
    ap=argparse.ArgumentParser();ap.add_argument('--train',required=True);ap.add_argument('--out',default='artifacts')
    args=ap.parse_args();rows,summary=clean(load(args.train),args.out)
    augmented,n=augment(rows);dump_jsonl(Path(args.out)/'augmented_train.jsonl',augmented)
    summary['augmented_rows']=n
    print(json.dumps(summary,indent=2))


In [ ]:
%%writefile score.py
"""Score triage predictions against labelled data.

    python score.py --gold data/dev.jsonl --pred my_dev_predictions.jsonl

Prediction file: one JSON object per line
    {"id": "dv-00001", "output": "<the raw text your model generated>"}

Scoring is strict:
  * "output" must be ONLY a JSON object (leading/trailing whitespace is ignored).
  * Every field must match exactly, including type (true is not 1, 129900 is not "129900").
  * A missing id or invalid JSON counts as wrong on every field.
"""
import argparse, json
from collections import defaultdict

KEYS = ["category", "priority", "amount_paise", "txn_date", "txn_id", "channel", "language", "needs_human"]


def load(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def same(a, b):
    return type(a) is type(b) and a == b


def parse(text):
    try:
        obj = json.loads(text.strip())
    except (json.JSONDecodeError, AttributeError):
        return None
    return obj if isinstance(obj, dict) else None


def score(gold_rows, pred_rows):
    preds = {p["id"]: p.get("output") for p in pred_rows}
    groups = defaultdict(lambda: {"n": 0, "valid": 0, "exact": 0, "field": defaultdict(int)})
    for g in gold_rows:
        gold = json.loads(g["messages"][-1]["content"])
        pred = parse(preds.get(g["id"]))
        names = ["all"] + ([f"slice={g['slice']}"] if "slice" in g else []) + [f"category={gold['category']}"]
        hits = {k: pred is not None and k in pred and same(pred[k], gold[k]) for k in KEYS}
        exact = pred is not None and all(hits.values()) and set(pred) == set(KEYS)
        for name in names:
            s = groups[name]
            s["n"] += 1
            s["valid"] += pred is not None
            s["exact"] += exact
            for k in KEYS:
                s["field"][k] += hits[k]
    out = {}
    for name, s in groups.items():
        n = s["n"]
        fields = {k: s["field"][k] / n for k in KEYS}
        out[name] = {"n": n, "json_valid": s["valid"] / n, "exact_match": s["exact"] / n,
                     "mean_field_acc": sum(fields.values()) / len(KEYS), "fields": fields}
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--gold", required=True)
    ap.add_argument("--pred", required=True)
    ap.add_argument("--json", action="store_true", help="print machine-readable JSON")
    a = ap.parse_args()
    res = score(load(a.gold), load(a.pred))
    if a.json:
        print(json.dumps(res, indent=1))
        return
    print(f"{'group':28} {'n':>5} {'valid':>7} {'exact':>7} {'fields':>7}")
    for name in sorted(res, key=lambda x: (x != "all", x)):
        r = res[name]
        print(f"{name:28} {r['n']:5d} {r['json_valid']:7.1%} {r['exact_match']:7.1%} {r['mean_field_acc']:7.1%}")
    print("\nper-field accuracy (all rows)")
    for k, v in res["all"]["fields"].items():
        print(f"  {k:14} {v:6.1%}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile training.py
"""Transparent PEFT training, evaluation and export for a free Colab T4."""
import copy
import gc
import hashlib
import importlib.metadata
import json
import math
import os
import random
import shutil
import time
from pathlib import Path

import numpy as np
import torch
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, GenerationConfig, Trainer, TrainingArguments, set_seed
from audit import augment, clean, dump_jsonl, load
from score import score

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
REVISION = '989aa7980e4cf806f80c7fef2b1adb7bc71aa306'
RUNS = {
    'A': {'data':'raw', 'four_bit':True, 'lr':2e-4},
    'B': {'data':'clean', 'four_bit':True, 'lr':2e-4},
    'C': {'data':'clean', 'four_bit':False, 'lr':2e-4},
    'D': {'data':'clean', 'four_bit':True, 'lr':1e-4},
    'E': {'data':'augmented', 'four_bit':True, 'lr':2e-4},
}


def environment():
    assert torch.cuda.is_available(), 'Training requires Colab GPU'
    assert 'T4' in torch.cuda.get_device_name(0), 'Only free Colab T4 is allowed'
    packages = {p:importlib.metadata.version(p) for p in ['torch','transformers','peft','accelerate','bitsandbytes','datasets','huggingface-hub','tokenizers','safetensors']}
    return {'gpu':torch.cuda.get_device_name(0),'gpu_bytes':torch.cuda.get_device_properties(0).total_memory,'cuda':torch.version.cuda,'packages':packages,'model_id':MODEL_ID,'revision':REVISION}


def load_base(four_bit):
    set_seed(42)
    kwargs = {'revision':REVISION,'torch_dtype':torch.float16,'device_map':{'':0},'attn_implementation':'sdpa','trust_remote_code':False}
    if four_bit:
        kwargs['quantization_config'] = BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16)
    return AutoModelForCausalLM.from_pretrained(MODEL_ID,**kwargs)


def encode_row(tokenizer,row):
    assert row['id'].startswith('tr-'), 'Dev/test must never enter tokenization for training'
    prefix = tokenizer.apply_chat_template(row['messages'][:2],tokenize=True,add_generation_prompt=True)
    full = tokenizer.apply_chat_template(row['messages'],tokenize=True,add_generation_prompt=False)
    assert full[:len(prefix)] == prefix, 'Chat template must preserve assistant-header prefix'
    end_id = tokenizer.convert_tokens_to_ids('<|im_end|>')
    end_position = full.index(end_id,len(prefix))
    # The template adds a separator newline after the assistant end token.
    # Supervise the complete response and end token, mask that separator too.
    labels = [-100]*len(prefix) + full[len(prefix):end_position+1] + [-100]*(len(full)-end_position-1)
    supervised = tokenizer.decode(full[len(prefix):],skip_special_tokens=False)
    assert supervised.startswith(row['messages'][-1]['content'])
    assert full[end_position] == end_id
    assert labels[end_position] == end_id
    assert all(value == -100 for value in labels[end_position+1:])
    return {'input_ids':full,'attention_mask':[1]*len(full),'labels':labels}


class TicketDataset(torch.utils.data.Dataset):
    def __init__(self, tokenizer, rows, maximum):
        self.items = [encode_row(tokenizer,r) for r in rows]
        assert all(len(item['input_ids'])<=maximum for item in self.items), 'No truncation allowed'
    def __len__(self):
        return len(self.items)
    def __getitem__(self,index):
        return self.items[index]


class AssistantCollator:
    def __init__(self,tokenizer):
        self.pad = tokenizer.pad_token_id
    def __call__(self,items):
        size=math.ceil(max(len(item['input_ids']) for item in items)/8)*8
        batch={key:[] for key in ('input_ids','attention_mask','labels')}
        for item in items:
            padding=size-len(item['input_ids'])
            batch['input_ids'].append(item['input_ids']+[self.pad]*padding)
            batch['attention_mask'].append(item['attention_mask']+[0]*padding)
            batch['labels'].append(item['labels']+[-100]*padding)
        return {key:torch.tensor(values,dtype=torch.long) for key,values in batch.items()}


def prepare(data_dir,out):
    data_dir,out=Path(data_dir),Path(out)
    out.mkdir(parents=True,exist_ok=True)
    raw=load(data_dir/'train.jsonl')
    cleaned,summary=clean(raw,out)
    augmented,n=augment(cleaned)
    dump_jsonl(out/'augmented_train.jsonl',augmented)
    summary['augmented_rows']=n
    (out/'audit_summary.json').write_text(json.dumps(summary,indent=2))
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=REVISION,trust_remote_code=False)
    assert tokenizer.pad_token_id is not None
    lengths=[len(encode_row(tokenizer,row)['input_ids']) for row in raw+cleaned+augmented]
    maximum=math.ceil(max(lengths)/128)*128
    stats={'p50':int(np.percentile(lengths,50)),'p95':int(np.percentile(lengths,95)),'p99':int(np.percentile(lengths,99)),'maximum_observed':max(lengths),'max_sequence_length':maximum,'no_truncation':True}
    (out/'token_lengths.json').write_text(json.dumps(stats,indent=2))
    (out/'environment.json').write_text(json.dumps(environment(),indent=2))
    print('Audit:',json.dumps(summary));print('Token lengths:',json.dumps(stats))
    return tokenizer,{'raw':raw,'clean':cleaned,'augmented':augmented},maximum


@torch.inference_mode()
def generate_rows(model,tokenizer,rows,path,rules=None,resume=False):
    """Only generation/decode/whitespace strip. No audit helpers or output repair."""
    model.eval()
    model.config.use_cache=True
    eos=model.generation_config.eos_token_id
    generation=GenerationConfig(do_sample=False,num_beams=1,max_new_tokens=200,eos_token_id=eos,pad_token_id=tokenizer.pad_token_id,use_cache=True)
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    predictions=[]
    if resume and path.exists():
        # Recover flushed records only. Never alter a model output or accept a
        # partial/broken JSONL record; validation must precede appending.
        saved=path.read_text()
        assert not saved or saved.endswith('\n'), 'Incomplete JSONL write: preserve and inspect the file before resuming'
        predictions=[json.loads(line) for line in saved.splitlines()]
        assert len(predictions)<=len(rows)
        assert [p['id'] for p in predictions]==[r['id'] for r in rows[:len(predictions)]]
        assert all(set(p)=={'id','output'} and isinstance(p['output'],str) for p in predictions)
        assert len({p['id'] for p in predictions})==len(predictions)
        print(f'{path.name}: retaining {len(predictions)} completed outputs',flush=True)
    completed=len(predictions)
    started=time.perf_counter()
    if not resume or not path.exists():
        path.write_text('')
    for index,row in enumerate(rows[completed:],start=completed):
        messages=copy.deepcopy([m for m in row['messages'] if m['role'] in {'system','user'}])
        assert [m['role'] for m in messages] == ['system','user']
        if rules is not None:
            messages[0]['content'] += '\n\n'+rules
        inputs=tokenizer.apply_chat_template(messages,tokenize=True,add_generation_prompt=True,return_tensors='pt').to(model.device)
        assert inputs.shape[1]+200 <= model.config.max_position_embeddings, 'Input exceeds model context; never truncate'
        ids=model.generate(input_ids=inputs,attention_mask=torch.ones_like(inputs),generation_config=generation)
        output=tokenizer.decode(ids[0,inputs.shape[1]:],skip_special_tokens=True).strip()
        prediction={'id':row['id'],'output':output}
        predictions.append(prediction)
        # Closing every record matters for the mounted Drive writer: a
        # terminated runtime lost an open partial baseline file despite flush.
        with path.open('a') as f:
            f.write(json.dumps(prediction,ensure_ascii=False)+'\n')
        if (index+1)%25==0:print(f'{path.name}: {index+1}/{len(rows)}; {(time.perf_counter()-started)/60:.1f} min',flush=True)
    return predictions


def evaluate(model,tokenizer,dev,out,rules=None,resume=False):
    out=Path(out)
    predictions=generate_rows(model,tokenizer,dev,out/'predictions_dev.jsonl',rules=rules,resume=resume)
    metrics=score(dev,predictions)
    (out/'dev_metrics.json').write_text(json.dumps(metrics,indent=2))
    print('Dev:',json.dumps(metrics['all']),flush=True)
    return metrics


def release(model):
    # Caller must discard its own reference before emptying the cache.
    del model
    gc.collect()
    torch.cuda.empty_cache()


def baseline(tokenizer,dev,out,rules):
    root=Path(out)
    model=load_base(False)
    for name,prompt_rules in [('baseline_fixed',None),('baseline_rules',rules)]:
        metrics=evaluate(model,tokenizer,dev,root/name,rules=prompt_rules,resume=True)
        (root/name/'result.json').write_text(json.dumps({'run':name,'four_bit':False,'metrics':metrics['all']},indent=2))
    del model;gc.collect();torch.cuda.empty_cache()


def train_run(name,tokenizer,rows,maximum,out,dev,max_steps=160):
    config=RUNS[name]
    destination=Path(out)/name
    destination.mkdir(parents=True,exist_ok=True)
    if (destination/'result.json').exists():
        print('Using saved completed run',name)
        return json.loads((destination/'result.json').read_text())
    set_seed(42)
    model=load_base(config['four_bit'])
    if config['four_bit']:
        model=prepare_model_for_kbit_training(model,use_gradient_checkpointing=True,gradient_checkpointing_kwargs={'use_reentrant':False})
    else:
        model.requires_grad_(False)
        model.enable_input_require_grads()
    model.config.use_cache=False
    lora=LoraConfig(r=16,lora_alpha=32,lora_dropout=.05,bias='none',task_type='CAUSAL_LM',target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
    model=get_peft_model(model,lora)
    trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
    assert 0 < trainable < 50_000_000
    assert all('lora_' in name for name,p in model.named_parameters() if p.requires_grad)
    dataset=TicketDataset(tokenizer,rows,maximum)
    kwargs=dict(output_dir=str(destination/'checkpoints'),max_steps=max_steps,per_device_train_batch_size=1,gradient_accumulation_steps=16,learning_rate=config['lr'],warmup_steps=8,lr_scheduler_type='cosine',optim='adamw_torch',weight_decay=0,fp16=True,bf16=False,gradient_checkpointing=True,gradient_checkpointing_kwargs={'use_reentrant':False},max_grad_norm=1.0,logging_steps=10,save_strategy='steps',save_steps=50,save_total_limit=1,report_to='none',seed=42,data_seed=42,group_by_length=True,dataloader_num_workers=0,remove_unused_columns=False)
    args=TrainingArguments(**kwargs)
    (destination/'configuration.json').write_text(json.dumps({'run':name,'model_id':MODEL_ID,'revision':REVISION,'training_arguments':args.to_dict(),'lora':lora.to_dict(),'four_bit':config['four_bit'],'quantization':{'type':'nf4','double_quantization':True,'compute_dtype':'float16'} if config['four_bit'] else None,'trainable_parameters':trainable},indent=2,default=lambda value: sorted(value) if isinstance(value,set) else str(value)))
    trainer=Trainer(model=model,args=args,train_dataset=dataset,data_collator=AssistantCollator(tokenizer),processing_class=tokenizer)
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    started=time.perf_counter()
    candidates=sorted((destination/'checkpoints').glob('checkpoint-*'),key=lambda p:int(p.name.split('-')[-1]))
    trainer.train(resume_from_checkpoint=str(candidates[-1]) if candidates else None)
    assert trainer.state.global_step == max_steps
    assert all(math.isfinite(step['loss']) for step in trainer.state.log_history if 'loss' in step), 'Non-finite training loss'
    torch.cuda.synchronize()
    minutes=(time.perf_counter()-started)/60
    peak_allocated=torch.cuda.max_memory_allocated()
    peak_reserved=torch.cuda.max_memory_reserved()
    (destination/'training_summary.json').write_text(json.dumps({'training_minutes':minutes,'peak_gpu_allocated_bytes':peak_allocated,'peak_gpu_reserved_bytes':peak_reserved,'max_steps':trainer.state.global_step},indent=2))
    model.save_pretrained(destination/'adapter',safe_serialization=True)
    tokenizer.save_pretrained(destination/'adapter')
    (destination/'training_log.json').write_text(json.dumps(trainer.state.log_history,indent=2))
    model.gradient_checkpointing_disable()
    metrics=evaluate(model,tokenizer,dev,destination)
    result={'run':name,**config,'seed':42,'train_rows':len(rows),'trainable_parameters':trainable,'max_steps':max_steps,'effective_batch_size':16,'max_sequence_length':maximum,'training_minutes':minutes,'peak_gpu_allocated_bytes':peak_allocated,'peak_gpu_reserved_bytes':peak_reserved,'metrics':metrics['all'],'model_id':MODEL_ID,'revision':REVISION}
    (destination/'result.json').write_text(json.dumps(result,indent=2))
    print('RESULT:',json.dumps(result),flush=True)
    del trainer,model,dataset;gc.collect();torch.cuda.empty_cache()
    return result


def select_best(out):
    results=[json.loads(path.read_text()) for path in Path(out).glob('[A-E]/result.json')]
    assert results,'No completed training runs'
    best=max(results,key=lambda r:(r['metrics']['exact_match'],r['metrics']['mean_field_acc'],-r['peak_gpu_allocated_bytes']))
    (Path(out)/'best_run.json').write_text(json.dumps(best,indent=2))
    return best


def export_best(tokenizer,dev,test,out,submission,resume_test=False):
    out,submission=Path(out),Path(submission)
    best=select_best(out)
    submission.mkdir(parents=True,exist_ok=True)
    provenance=submission/'selected_run.json'
    if resume_test and (submission/'predictions_test.jsonl').exists():
        previous=json.loads(provenance.read_text())
        assert all(previous[k]==best[k] for k in ['run','model_id','revision','four_bit']), 'Saved test outputs belong to another run'
        def digest(path):
            h=hashlib.sha256()
            with path.open('rb') as f:
                for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
            return h.hexdigest()
        assert digest(submission/'adapter/adapter_model.safetensors')==digest(out/best['run']/'adapter/adapter_model.safetensors'), 'Adapter provenance mismatch'
    provenance.write_text(json.dumps(best,indent=2))
    shutil.copytree(out/best['run']/'adapter',submission/'adapter',dirs_exist_ok=True)
    shutil.copy2(out/'cleaning_log.csv',submission/'cleaning_log.csv')
    model=PeftModel.from_pretrained(load_base(best['four_bit']),submission/'adapter')
    predictions=generate_rows(model,tokenizer,test,submission/'predictions_test.jsonl',resume=resume_test)
    assert len(predictions)==len(test)==400
    assert {p['id'] for p in predictions}=={r['id'] for r in test}
    del model;gc.collect();torch.cuda.empty_cache()
    model=PeftModel.from_pretrained(load_base(best['four_bit']),submission/'adapter')
    chosen=random.Random(42).sample(test,10)
    rerun=generate_rows(model,tokenizer,chosen,out/'reproduction_sample.jsonl')
    lookup={p['id']:p['output'] for p in predictions}
    assert all(p['output']==lookup[p['id']] for p in rerun),'Adapter reload/sample reproduction mismatch'
    (out/'reproduction_check.json').write_text(json.dumps({'sample_size':10,'all_outputs_match':True,'seed':42},indent=2))
    del model;gc.collect();torch.cuda.empty_cache()
    return best


In [ ]:
%%writefile resume_submission.py
"""Resume baseline inference and test export from completed Drive runs.

Run only after free Colab T4 access returns. This entry point never calls training.
Existing baseline JSONL records must form an intact prefix of the dev IDs.
"""
import argparse
import json
import shutil
from pathlib import Path

from transformers import AutoTokenizer
from training import MODEL_ID, REVISION, baseline, environment, export_best, load


def resume(data, out, submission, schema):
    print('Resume environment:', json.dumps(environment()), flush=True)
    for name in 'ABCDE':
        result=json.loads((out/name/'result.json').read_text())
        assert result['max_steps']==160 and result['metrics']['n']==200
        assert result['model_id']==MODEL_ID and result['revision']==REVISION
        assert (out/name/'adapter/adapter_model.safetensors').exists()
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=REVISION,trust_remote_code=False)
    dev=load(data/'dev.jsonl');test=load(data/'test_inputs.jsonl')
    assert len(dev)==200 and len(test)==400
    # Complete saved baselines; the fixed baseline's 200 outputs are rescored,
    # while only missing schema-baseline IDs are generated.
    baseline(tokenizer,dev,out,schema.read_text())
    best=export_best(tokenizer,dev,test,out,submission,resume_test=True)
    results=submission/'results'
    results.mkdir(exist_ok=True)
    for path in out.rglob('*.json'):
        if 'checkpoints' not in path.parts and 'adapter' not in path.parts:
            target=results/path.relative_to(out)
            target.parent.mkdir(parents=True,exist_ok=True)
            shutil.copy2(path,target)
    for name in ['A','B','C','D','E','baseline_fixed','baseline_rules']:
        target=results/name/'predictions_dev.jsonl'
        target.parent.mkdir(parents=True,exist_ok=True)
        shutil.copy2(out/name/'predictions_dev.jsonl',target)
    print('RESUMED EVALUATION AND TEST EXPORT COMPLETE',json.dumps(best),flush=True)
    return best


if __name__=='__main__':
    ap=argparse.ArgumentParser()
    ap.add_argument('--data',type=Path,required=True)
    ap.add_argument('--out',type=Path,required=True)
    ap.add_argument('--submission',type=Path,required=True)
    ap.add_argument('--schema',type=Path,required=True)
    args=ap.parse_args()
    resume(args.data,args.out,args.submission,args.schema)


In [ ]:
from resume_submission import resume
best=resume(DATA,OUT,SUBMISSION,ROOT/'private/candidate_pack/SCHEMA.md')
shutil.copy2(ROOT/'training.py',SUBMISSION/'executed_training.py')
archive=shutil.make_archive('/content/nimbuspay_resumed_submission','zip',SUBMISSION)
print('Saved private submission:',SUBMISSION)
files.download(archive)
